# Robust Evaluation – 10 Stratified 80/20 Splits

In this notebook we perform a fair evaluation of the Random Forest models using the 
hyperparameters previously obtained via Bayesian Optimization.

Instead of relying on a single train/test split, we perform:

- 10 independent 80/20 stratified splits
- Retrain models from scratch on each split
- Evaluate on each corresponding test set
- Average performance metrics across splits

This removes the effect of a potentially favorable split and provides a robust estimate of performance.

We evaluate both approaches:

1. Independent Binary classifiers
2. Label Power Set (LPS) classifier

Metrics computed 
- Accuracy (ACC)
- Hamming Loss (HL)
- Weighted F1 (WF1)

For LPS:
Predicted patterns are decomposed into individual binary labels before computing metrics.

# Imports and Configuration

In [ ]:
import pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, hamming_loss, f1_score
import time
from tqdm import tqdm
import itertools

RANDOM_SEEDS = list(range(10))
TEST_SIZE = 0.20

## Utility Functions

In [2]:
def multilabel_weighted_f1(y_true, y_pred):
    total = 0
    for col in range(y_true.shape[1]):
        total += f1_score(y_true[:, col], y_pred[:, col], average="weighted")
    return total / y_true.shape[1]

## Load dataset

In [3]:
with open("../data/DRIAMS_A_AMR_paper_replication.pkl", "rb") as f:
    payload = pickle.load(f)

X_raw = payload["data"]
amr_raw = payload["amr"]
antibiotics = payload["antibiotics"]
labels_raw = payload["label"]

df_features = pd.DataFrame(X_raw)
df_amr = pd.DataFrame(amr_raw, columns=antibiotics)
df_species = pd.DataFrame(labels_raw, columns=["species"])

full_df = pd.concat([df_features, df_amr, df_species], axis=1)
full_df = full_df.drop_duplicates()

n_features = X_raw.shape[1]
feature_cols = list(full_df.columns[:n_features])

## Hyperparameters (From Bayesian Optimization)

In [4]:
hyperparams_binary = {
    "Staphylococcus_Aureus": {
        "Oxacillin": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 42},
        "Clindamycin": {'bootstrap': True, 'max_depth': 5, 'min_samples_leaf': 9, 'n_estimators': 1},
        "Fusidic acid": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 447},
    },
    "Escherichia_Coli": {
        "Ciprofloxacin": {'bootstrap': False, 'max_depth': 9, 'min_samples_leaf': 1, 'n_estimators': 1},
        "Ceftriaxone": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 6},
        "Piperacillin-Tazobactam": {'bootstrap': False, 'max_depth': 5, 'min_samples_leaf': 2, 'n_estimators': 334},
        "Cefepime": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 3, 'n_estimators': 5},
    },
    "Klebsiella_Pneumoniae": {
        "Ciprofloxacin": {'bootstrap': False, 'max_depth': 7, 'min_samples_leaf': 10, 'n_estimators': 3},
        "Ceftriaxone": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 4, 'n_estimators': 17},
        "Imipenem": {'bootstrap': True, 'max_depth': 7, 'min_samples_leaf': 7, 'n_estimators': 647},
        "Meropenem": {'bootstrap': True, 'max_depth': 7, 'min_samples_leaf': 7, 'n_estimators': 647},
    },
    "Pseudomonas_Aeruginosa": {
        "Ciprofloxacin": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 520},
        "Imipenem": {'bootstrap': False, 'max_depth': 5, 'min_samples_leaf': 10, 'n_estimators': 2},
        "Meropenem": {'bootstrap': True, 'max_depth': 7, 'min_samples_leaf': 1, 'n_estimators': 9},
    }
}

hyperparams_lps = {
    "Staphylococcus_Aureus": {'bootstrap': True, 'max_depth': 10, 'min_samples_leaf': 10, 'n_estimators': 2},
    "Escherichia_Coli": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 2},
    "Klebsiella_Pneumoniae": {'bootstrap': False, 'max_depth': 8, 'min_samples_leaf': 1, 'n_estimators': 4},
    "Pseudomonas_Aeruginosa": {'bootstrap': False, 'max_depth': 10, 'min_samples_leaf': 2, 'n_estimators': 341},
}

## Evaluation Loop – 10 Stratified Splits

For each species:
- Remove rare patterns (<10 samples)
- Perform 10 independent 80/20 splits (stratified by pattern)
- Train binary models
- Train LPS model
- Decompose LPS predictions into multi-label
- Compute ACC, HL, WF1
- Average across splits

In [7]:
species_antibiotics = {
    "Staphylococcus_Aureus": ["Oxacillin", "Clindamycin", "Fusidic acid"],
    "Escherichia_Coli": ["Ciprofloxacin", "Ceftriaxone", "Piperacillin-Tazobactam", "Cefepime"],
    "Klebsiella_Pneumoniae": ["Ciprofloxacin", "Ceftriaxone", "Imipenem", "Meropenem"],
    "Pseudomonas_Aeruginosa": ["Ciprofloxacin", "Imipenem", "Meropenem"],
}

results = {}

global_start = time.time()

for species, ab_list in species_antibiotics.items():

    print("\n" + "="*70, flush=True)
    print(f"Processing species: {species}", flush=True)
    print("="*70, flush=True)

    species_start = time.time()

    df_sp = full_df[full_df["species"] == species]
    print(f"Initial samples: {df_sp.shape[0]}", flush=True)

    sp_df = df_sp[feature_cols + ab_list].dropna()
    print(f"After NaN removal: {sp_df.shape[0]}", flush=True)

    X = sp_df.iloc[:, :n_features].to_numpy()
    y_multi = sp_df[ab_list].to_numpy().astype(int)

    patterns = np.array(["".join(map(str, row)) for row in y_multi])
    counts = pd.Series(patterns).value_counts()
    valid = counts[counts >= 10].index
    mask = np.isin(patterns, valid)

    X = X[mask]
    y_multi = y_multi[mask]
    patterns = patterns[mask]

    print(f"After rare-pattern filtering (<10 removed): {X.shape[0]}", flush=True)
    print(f"Unique patterns kept: {len(np.unique(patterns))}", flush=True)

    metrics_bin = []
    metrics_lps = []

    for seed in tqdm(RANDOM_SEEDS, desc=f"{species} splits"):

        split_start = time.time()

        train_idx, test_idx = train_test_split(
            np.arange(X.shape[0]),
            test_size=TEST_SIZE,
            random_state=seed,
            stratify=patterns
        )

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y_multi[train_idx], y_multi[test_idx]
        patterns_train = patterns[train_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # ===============================
        # Binary Models
        # ===============================
        preds_bin = []

        for i, ab in enumerate(ab_list):
            params = hyperparams_binary[species][ab]
            model = RandomForestClassifier(**params, random_state=0)
            model.fit(X_train, y_train[:, i])
            preds_bin.append(model.predict(X_test))

        preds_bin = np.array(preds_bin).T

        acc_bin = accuracy_score(y_test, preds_bin)
        ham_bin = hamming_loss(y_test, preds_bin)
        f1_bin = multilabel_weighted_f1(y_test, preds_bin)

        metrics_bin.append((acc_bin, ham_bin, f1_bin))

        # ===============================
        # LPS Model
        # ===============================
        le = LabelEncoder()
        y_train_lps = le.fit_transform(patterns_train)

        model_lps = RandomForestClassifier(
            **hyperparams_lps[species],
            random_state=0
        )
        model_lps.fit(X_train, y_train_lps)

        pred_lps = model_lps.predict(X_test)
        decoded = le.inverse_transform(pred_lps)
        preds_multi = np.array([[int(c) for c in s] for s in decoded])

        acc_lps = accuracy_score(y_test, preds_multi)
        ham_lps = hamming_loss(y_test, preds_multi)
        f1_lps = multilabel_weighted_f1(y_test, preds_multi)

        metrics_lps.append((acc_lps, ham_lps, f1_lps))

        split_time = time.time() - split_start
        print(f"Seed {seed} done in {split_time:.2f} sec", flush=True)

    results[species] = {
        "binary_mean": np.mean(metrics_bin, axis=0),
        "binary_std": np.std(metrics_bin, axis=0),
        "lps_mean": np.mean(metrics_lps, axis=0),
        "lps_std": np.std(metrics_lps, axis=0),
    }

    species_time = time.time() - species_start
    print(f"\nFinished {species} in {species_time/60:.2f} minutes", flush=True)

total_time = time.time() - global_start
print("\n" + "="*70)
print(f"TOTAL EXECUTION TIME: {total_time/60:.2f} minutes")
print("="*70)

results
results


Processing species: Staphylococcus_Aureus
Initial samples: 3791
After NaN removal: 3556
After rare-pattern filtering (<10 removed): 3556
Unique patterns kept: 8


Staphylococcus_Aureus splits:   0%|          | 0/10 [00:00<?, ?it/s]

Seed 0 done in 49.67 sec


Staphylococcus_Aureus splits:  10%|█         | 1/10 [00:49<07:27, 49.67s/it]

Seed 1 done in 49.45 sec


...

Pseudomonas_Aeruginosa splits:  90%|█████████ | 9/10 [07:55<00:52, 52.79s/it]

Seed 9 done in 52.49 sec


Pseudomonas_Aeruginosa splits: 100%|██████████| 10/10 [08:47<00:00, 52.76s/it]


Finished Pseudomonas_Aeruginosa in 8.79 minutes

TOTAL EXECUTION TIME: 26.88 minutes


{'Staphylococcus_Aureus': {'binary_mean': array([0.68960674, 0.12762172, 0.83181567]),
  'binary_std': array([0.01175084, 0.00492442, 0.00408442]),
  'lps_mean': array([0.67738764, 0.13731273, 0.81975393]),
  'lps_std': array([0.01064177, 0.00369257, 0.00505811])},
 'Escherichia_Coli': {'binary_mean': array([0.57360515, 0.18819742, 0.76702407]),
  'binary_std': array([0.01223552, 0.00490843, 0.00715785]),
  'lps_mean': array([0.61480687, 0.19447425, 0.75929642]),
  'lps_std': array([0.00654418, 0.00437651, 0.00613156])},
 'Klebsiella_Pneumoniae': {'binary_mean': array([0.78908766, 0.08094812, 0.89065807]),
  'binary_std': array([0.00411449, 0.00220914, 0.00439055]),
  'lps_mean': array([0.78944544, 0.08358676, 0.88850605]),
  'lps_std': array([0.00536971, 0.00217676, 0.00369138])},
 'Pseudomonas_Aeruginosa': {'binary_mean': array([0.80066372, 0.11976401, 0.8282222 ]),
  'binary_std': array([0.00572235, 0.00209107, 0.00218162]),
  'lps_mean': array([0.8050885 , 0.11880531, 0.8268649 ]),

In [8]:
import pandas as pd

rows = []

for species, metrics in results.items():
    acc_bin, hl_bin, wf1_bin = metrics["binary_mean"]
    acc_lps, hl_lps, wf1_lps = metrics["lps_mean"]

    rows.append({
        "Bacteria": species,
        "Model": "RF",
        "ACC (Single-label)": acc_bin,
        "ACC (Multi-label)": acc_lps,
        "HL (Single-label)": hl_bin,
        "HL (Multi-label)": hl_lps,
        "WF1 (Single-label)": wf1_bin,
        "WF1 (Multi-label)": wf1_lps,
    })

df_results = pd.DataFrame(rows)

# Formato tipo paper (3 decimales)
df_results = df_results.round(3)

df_results

,Bacteria,Model,ACC (Single-label),ACC (Multi-label),HL (Single-label),HL (Multi-label),WF1 (Single-label),WF1 (Multi-label)
0,Staphylococcus_Aureus,RF,0.690,0.677,0.128,0.137,0.832,0.820
1,Escherichia_Coli,RF,0.574,0.615,0.188,0.194,0.767,0.759
2,Klebsiella_Pneumoniae,RF,0.789,0.789,0.081,0.084,0.891,0.889
3,Pseudomonas_Aeruginosa,RF,0.801,0.805,0.120,0.119,0.828,0.827


## Classifier Chain Evaluation with Random Forest (All Possible Label Orders)

To further investigate whether resistance predictions for certain antibiotics could improve the prediction of others, we implemented a Classifier Chain (CC) strategy using Random Forest (RF) models.

Unlike independent binary models, where each antibiotic is predicted separately, the Classifier Chain approach models potential interdependencies between resistance phenotypes. In this framework, antibiotics are arranged in a specific order, and predictions are made sequentially. For each antibiotic in the chain:

- The model receives as input the original MALDI-TOF features.
- Additionally, it incorporates the previously predicted resistance labels as extra input features.

During training, true labels from earlier positions in the chain are appended to the feature matrix (teacher forcing). During testing, predicted labels from earlier steps are used instead. This allows the model to explicitly learn conditional dependencies between antibiotics.

Since the order of antibiotics in the chain may influence performance, we evaluated **all possible permutations of antibiotic orders** for each bacterial species. For example:

- 3 antibiotics → 3! = 6 possible orders  
- 4 antibiotics → 4! = 24 possible orders  

For each order, we performed:

- 10 independent 80/20 train-test split
- Evaluation on the corresponding test sets

Performance metrics were computed as described in the main evaluation protocol:

- Accuracy (ACC)
- Hamming Loss (HL)
- Weighted F1 score (WF1)

Results were averaged across the 10 splits, and the best-performing chain order was selected based on the mean WF1 score, which was defined as the primary metric due to its robustness to class imbalance.

This approach allows us to assess whether explicitly modeling inter-antibiotic dependencies improves predictive performance compared to:

1. Independent binary classification  
2. Label Power Set (LPS) multi-class modeling  

By evaluating all possible chain orders, we ensure that conclusions are not dependent on an arbitrary label ordering.

In [ ]:
def fit_predict_classifier_chain_rf(
    X_train, y_train, X_test, ab_list, order, hyperparams_binary, species
):

    n_labels = len(ab_list)
    ab_to_idx = {ab: i for i, ab in enumerate(ab_list)}
    order_idx = [ab_to_idx[ab] for ab in order]

    y_pred_test = np.zeros((X_test.shape[0], n_labels), dtype=int)

    for step, label_idx in enumerate(order_idx):

        ab_name = ab_list[label_idx]
        params = hyperparams_binary[species][ab_name]

        if step > 0:
            prev_indices = order_idx[:step]
            Xtr_aug = np.hstack([X_train, y_train[:, prev_indices]])
            Xte_aug = np.hstack([X_test,  y_pred_test[:, prev_indices]])
        else:
            Xtr_aug = X_train
            Xte_aug = X_test

        model = RandomForestClassifier(**params, random_state=0)
        model.fit(Xtr_aug, y_train[:, label_idx])

        y_pred_test[:, label_idx] = model.predict(Xte_aug).astype(int)

    return y_pred_test


# ============================================================
# MAIN – SINGLE SPLIT PER SPECIES
# ============================================================

chain_results = {}
RANDOM_STATE_SPLIT = 42
TEST_SIZE = 0.20

global_start = time.time()

for species, ab_list in species_antibiotics.items():

    print("\n" + "="*80)
    print(f"SPECIES: {species}")
    print("="*80)

    df_sp = full_df[full_df["species"] == species]
    print(f"Initial samples: {df_sp.shape[0]}")

    sp_df = df_sp[feature_cols + ab_list].dropna()
    print(f"After NaN removal: {sp_df.shape[0]}")

    X = sp_df.iloc[:, :n_features].to_numpy()
    y_multi = sp_df[ab_list].to_numpy().astype(int)

    # Filtrado patrones raros
    patterns = np.array(["".join(map(str, row)) for row in y_multi])
    counts = pd.Series(patterns).value_counts()
    valid = counts[counts >= 10].index
    mask = np.isin(patterns, valid)

    X = X[mask]
    y_multi = y_multi[mask]
    patterns = patterns[mask]

    print(f"After rare-pattern filtering: {X.shape[0]}")
    print(f"Unique patterns: {len(np.unique(patterns))}")

    if X.shape[0] == 0:
        print("No samples left. Skipping.")
        continue

    # ==========================
    # SINGLE STRATIFIED SPLIT
    # ==========================
    train_idx, test_idx = train_test_split(
        np.arange(X.shape[0]),
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE_SPLIT,
        stratify=patterns
    )

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_multi[train_idx], y_multi[test_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # ==========================
    # TEST ALL ORDERS
    # ==========================
    all_orders = list(itertools.permutations(ab_list))
    print(f"Total orders: {len(all_orders)}")

    order_metrics = []

    for order in tqdm(all_orders, desc=f"{species} orders"):

        y_pred = fit_predict_classifier_chain_rf(
            X_train, y_train, X_test,
            ab_list, order,
            hyperparams_binary, species
        )

        acc = accuracy_score(y_test, y_pred)
        hl  = hamming_loss(y_test, y_pred)
        wf1 = multilabel_weighted_f1(y_test, y_pred)

        order_metrics.append({
            "order": order,
            "ACC": acc,
            "HL": hl,
            "WF1": wf1
        })

    df_orders = pd.DataFrame(order_metrics).sort_values("WF1", ascending=False).reset_index(drop=True)

    print("\nBEST ORDER (by WF1):")
    best = df_orders.iloc[0]
    print(f"Order: {best['order']}")
    print(f"ACC: {best['ACC']:.3f}")
    print(f"HL : {best['HL']:.3f}")
    print(f"WF1: {best['WF1']:.3f}")

    chain_results[species] = df_orders

total_time = time.time() - global_start
print("\n" + "="*80)
print(f"TOTAL EXECUTION TIME: {total_time/60:.2f} minutes")
print("="*80)


SPECIES: Staphylococcus_Aureus
Initial samples: 3791
After NaN removal: 3556
After rare-pattern filtering: 3556
Unique patterns: 8
Total orders: 6


Staphylococcus_Aureus orders: 100%|██████████| 6/6 [04:53<00:00, 48.88s/it]



BEST ORDER (by WF1):
Order: ('Oxacillin', 'Fusidic acid', 'Clindamycin')
ACC: 0.688
HL : 0.128
WF1: 0.836

SPECIES: Escherichia_Coli
Initial samples: 4990
After NaN removal: 4663
After rare-pattern filtering: 4659
Unique patterns: 12
Total orders: 24


Escherichia_Coli orders:  83%|████████▎ | 20/24 [08:22<01:40, 25.10s/it]